In [144]:
import pandas as pd
import numpy as np

np.set_printoptions(threshold=np.inf)

In [145]:
# Loading guest data from the restaurant
guest_df = pd.read_csv("../data/WEEVA_GUESTS.csv")
print(guest_df.head(1))

# Rename the column
guest_df.rename(columns={"DATE" : 'Date'}, inplace=True)

# Convert to datetime
guest_df["Date"] = pd.to_datetime(guest_df["Date"])


print(guest_df.head(1))

        DATE GUESTS
0  11/1/2018      0
        Date GUESTS
0 2018-11-01      0


In [146]:
# Loading weather data by time frame
weather_nov_2018_may_2021_df       = pd.read_csv("../data/Groningen 2018-11-01 to 2021-05-31.csv")
weather_june_2021_december_2023_df = pd.read_csv("../data/Groningen 2021-06-01 to 2023-12-31.csv")
weather_jan_2024_april_2025_df     = pd.read_csv("../data/Groningen 2024-01-01 to 2025-04-28.csv")

# Combining the weather datasets into one dataframe
weather_df = pd.concat([weather_nov_2018_may_2021_df, 
                        weather_june_2021_december_2023_df, 
                        weather_jan_2024_april_2025_df], 
                        ignore_index=True)

# Rename the column
weather_df.rename(columns={"datetime" : 'Date'}, inplace=True)

# Convert to datetime
weather_df["Date"] = pd.to_datetime(weather_df["Date"])

irrelevant_features = ['name', 'dew', 'precipcover', 'snow', 'snowdepth',
                        'winddir', 'sealevelpressure', 'visibility', 
                        'solarenergy', 'severerisk', 'sunrise', 'sunset',
                        'moonphase', 'description', 'stations', 'icon', 
                        'conditions']
weather_df = weather_df.drop(columns=irrelevant_features)

# Create boolean columns to encode preciptype
weather_df['rain'] = weather_df['preciptype'].str.contains('rain', na=False)\
                                                                .astype(int)
weather_df['snow'] = weather_df['preciptype'].str.contains('snow', na=False)\
                                                                .astype(int)

# Drop preciptype column
weather_df = weather_df.drop(columns='preciptype')

print(weather_df.columns)
print(weather_df.head())
print(weather_df.shape)

Index(['Date', 'tempmax', 'tempmin', 'temp', 'feelslikemax', 'feelslikemin',
       'feelslike', 'humidity', 'precip', 'precipprob', 'windgust',
       'windspeed', 'cloudcover', 'solarradiation', 'uvindex', 'rain', 'snow'],
      dtype='object')
        Date  tempmax  tempmin  temp  feelslikemax  feelslikemin  feelslike  \
0 2018-11-01     12.9      7.2   9.5          12.9           4.6        8.0   
1 2018-11-02     11.0      1.7   8.2          11.0          -1.1        6.7   
2 2018-11-03     10.0      0.6   4.6          10.0          -1.9        2.7   
3 2018-11-04     10.7     -0.4   5.2          10.7          -2.5        3.4   
4 2018-11-05     10.1      5.9   8.8          10.1           4.3        7.9   

   humidity  precip  precipprob  windgust  windspeed  cloudcover  \
0      83.5   2.532         100      35.0       16.3        21.5   
1      85.6   1.095         100      49.4       22.5        40.6   
2      89.2   0.000           0      30.9       15.5         6.1   
3     

In [147]:
# Loading data on holidays and calendar dates
school_holidays_df           = pd.read_csv("../data/groningen_school_holidays_boolean.csv")
public_holidays_groningen_df = pd.read_csv("../data/public_holidays_2018_2025.csv")
public_holidays_germany_df   = pd.read_csv("../data/public_holidays_germany_2018_2025.csv")
calendar_df                  = pd.read_csv("../data/dates_with_weekdays.csv")

In [148]:
# Convert calendar_df["Date"] to pd.to_datetime
calendar_df["Date"] = pd.to_datetime(calendar_df["Date"])

# Encode Day of the week into separate columns
dow_dummies = pd.get_dummies(calendar_df['DayOfWeek'], prefix='is', dtype=int)
calendar_df = pd.concat([calendar_df, dow_dummies], axis=1)

# Drop DayOfWeek and IsWeekend
calendar_df = calendar_df.drop(columns=['DayOfWeek', 'IsWeekend']) 

print(calendar_df.head(1))

        Date  is_Friday  is_Monday  is_Saturday  is_Sunday  is_Thursday  \
0 2018-11-01          0          0            0          0            1   

   is_Tuesday  is_Wednesday  
0           0             0  


In [149]:
school_holidays_df.tail()
school_holidays_df.shape
# school_holidays_df.dtypes

(3035, 2)

In [150]:
school_holidays_df["Date"] = pd.to_datetime(school_holidays_df["Date"])

start_date = '2018-11-01'
end_date = '2025-04-28'

# Filter to keep only dates within the desired range
school_holidays_df = school_holidays_df[
    (school_holidays_df['Date'] >= start_date) &
    (school_holidays_df['Date'] <= end_date)
]

# Remove duplicate dates, keeping the last occurrence
school_holidays_df = school_holidays_df.drop_duplicates(subset='Date', keep='last')

school_holidays_bool_df = pd.DataFrame(school_holidays_df)
# Convert 'Yes'/'No' to True/False in a specific column (e.g., 'IsHoliday')
school_holidays_bool_df['IsHoliday'] = school_holidays_bool_df['IsHoliday'].map({"Yes": 1, "No": 0})

school_holidays_bool_df.shape

C:\Users\Matei\AppData\Local\Temp\ipykernel_27440\4025799754.py:1: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  school_holidays_df["Date"] = pd.to_datetime(school_holidays_df["Date"])


(2371, 2)

In [151]:
# 9 entries out of our date range for groningen
# public_holidays_groningen_df.head(50)
# public_holidays_groningen_df.tail(20)

public_holidays_groningen_df["Holiday"].unique()
# Only 9 unique holidays, but inconsistant naming () 
# e.g: 'Koningsdag (National Holiday)' / 'Koningsdag (National Day)'
# public_holidays_groningen_df["Holiday"].unique().shape

public_holidays_groningen_df["Holiday"] = public_holidays_groningen_df\
                                                       ['Holiday'].str.strip()

# public_holidays_groningen_df["Holiday"].unique()

name_map = {
    "New Year": "New Year's Day",
    "Koningsdag (National Holiday)": "King\'s Day",
     "Koningsdag (National Day)": "King\'s Day",
     "St. Stephen's Day": "Second Christmas Day"
}

# Solve inconsistant naming
public_holidays_groningen_df["Holiday"] = public_holidays_groningen_df\
                                                 ['Holiday'].replace(name_map)

public_holidays_groningen_df["Holiday"].unique()
# # Only 7 unique holidays after filtering
# public_holidays_groningen_df["Holiday"].unique().shape

# public_holidays_groningen_df.head()


array(["New Year's Day", 'Easter Monday', "King's Day", 'Ascension Day',
       'Whit Monday', 'Christmas', 'Second Christmas Day'], dtype=object)

In [152]:
public_holidays_germany_df["Holiday"].unique()

public_holidays_germany_df["Holiday"] = public_holidays_germany_df\
                                                       ['Holiday'].str.strip()

name_map = {
    "New Years Day": "New Year's Day",
    "Christmas Day": "Christmas",
    "Boxing Day": "Second Christmas Day"
}

# Solve inconsistant naming
public_holidays_germany_df["Holiday"] = public_holidays_germany_df\
                                                 ['Holiday'].replace(name_map)

public_holidays_germany_df["Holiday"].unique()


array(["New Year's Day", 'Good Friday', 'Easter Monday', 'May Day',
       'Ascension Day', 'Whit Monday', 'Day of German Unity', 'Christmas',
       'Second Christmas Day'], dtype=object)

In [153]:
# Setting the date column to the right data type
public_holidays_groningen_df["Date"] = pd.to_datetime(
                                        public_holidays_groningen_df["Date"],
                                        format="%d.%m.%Y")
public_holidays_germany_df["Date"] = pd.to_datetime(
                                        public_holidays_germany_df["Date"],
                                        format="%d.%m.%Y")

# Combine all the holidays
combined_holidays_df = pd.concat([public_holidays_groningen_df,
                                   public_holidays_germany_df],
                                    ignore_index=True)

combined_holidays_df["Holiday"].unique().shape

combined_holidays_df['is_holiday'] = 1


# Drop duplicates
combined_holidays_df = combined_holidays_df.pivot_table(
    index='Date',
    columns='Holiday',
    values='is_holiday',
    fill_value=0
).reset_index()

combined_holidays_df.shape

# Defining a range of dates for the full holiday dataframe
date_range = pd.date_range(start='2018-11-01', end='2025-04-28', freq='D')

# Create a new DataFrame with that full date range
full_date_range_df = pd.DataFrame({'Date': date_range})

# Merge on Date — left join to preserve full date range
merged_df = full_date_range_df.merge(combined_holidays_df,
                                     on='Date',
                                     how='left')

# # Fill NaN's 
final_holiday_df = merged_df.fillna('0').astype({col: 'int' for col in \
                                                     merged_df.columns \
                                                        if col != 'Date'})

final_holiday_df.head()
final_holiday_df.columns




Index(['Date', 'Ascension Day', 'Christmas', 'Day of German Unity',
       'Easter Monday', 'Good Friday', 'King's Day', 'May Day',
       'New Year's Day', 'Second Christmas Day', 'Whit Monday'],
      dtype='object')

In [154]:
print(calendar_df.head(1))
print(calendar_df.tail(1))


        Date  is_Friday  is_Monday  is_Saturday  is_Sunday  is_Thursday  \
0 2018-11-01          0          0            0          0            1   

   is_Tuesday  is_Wednesday  
0           0             0  
           Date  is_Friday  is_Monday  is_Saturday  is_Sunday  is_Thursday  \
2370 2025-04-28          0          1            0          0            0   

      is_Tuesday  is_Wednesday  
2370           0             0  


In [155]:
# Loading data on number of items ordered in the restaurant
course_data_2018_2022_df = pd.read_csv("../data/Weeva_data_2018-2022.csv")
course_data_2023_2025_df = pd.read_csv("../data/Weeva_Data_2023-x.csv")

course_data_df = pd.concat([course_data_2018_2022_df, 
                            course_data_2023_2025_df], 
                            ignore_index=True)

# Setting the date column to the right data type
course_data_df["Date"] = pd.to_datetime(course_data_df["Date"],
                                        format="%d-%m-%Y")

article_map = {
    "broodplankje": "art_broodplankje",
    "captain.*dinner": "art_captain_dinner",
    "(?<!\w\s)cr.me.*br.l.e(?!\s)": "art_creme_brulee",
    "dame.*blanche(?!\s)": "art_dame_blanche",
    "sliptong.*meuni.re": "art_sliptong",
    "garnalen.*cocktail": "art_garnalen_cocktail",
    "bloedworst": "art_bloedworst",
    "olijven": "art_olijven",
    "kaasplankje(?!\s)": "art_kaasplankje",
    "(?<!\w\s)kalfslever": "art_kalfslever",
    "koffie.*compleet(?!\s)": "art_koffie_compleet",
    "groningse poffert": "art_poffert",
    "runder.*carpaccio": "art_carpaccio",
    "sat.*spies(?!\s)": "art_sate_spies",
    "schnitzel": "art_schnitzel",
    "sorbet.*weeva": "art_sorbet",
    "andijvie stamppot": "art_stamppot",
    "vers.*markt": "art_vers_van_de_markt",
    "weeva.*gehaktbal(?!\s)": "art_gehaktbal",
    "weeva.*spareribs(?!\s)": "art_spareribs",
    "tournedos(?!\s)": "art_tournedos",
    "zalmfilet(?!\s)": "art_zalmfilet",
}

# Lower case
course_data_df["Article"] = course_data_df["Article"].str.lower()

# Solve inconsistant naming
course_data_df["Article"] = course_data_df["Article"].replace(article_map,\
                                                               regex=True)

# Keep only rows where the Article value is one of the mapped values
valid_articles = set(article_map.values())
course_data_df = course_data_df[course_data_df["Article"].\
                                                        isin(valid_articles)]

# print(course_data_df["Article"].unique())
# print(course_data_df.head())

course_data_df = course_data_df.pivot_table(
    index="Date",
    columns="Article",
    values="Sold articles amount",
    fill_value=0,
    aggfunc="sum",
    ).reset_index()




In [156]:
print(course_data_df.head(1)["Date"])

print(course_data_df.tail(1)["Date"])

print(f"{course_data_df.shape=}")


print("First date in final_holiday_df =", final_holiday_df["Date"].head(1))
print("Last date in final_holiday_df  =", final_holiday_df["Date"].tail(1))

print(f"{final_holiday_df.shape=}")

print(final_holiday_df["Date"].unique())


course_dates = set(course_data_df["Date"])
holiday_dates = set(final_holiday_df["Date"])

# Dates in course_data_df but not in final_holiday_df
only_in_course = course_dates - holiday_dates
print("Dates in course_data_df but NOT in final_holiday_df:")
print(sorted(only_in_course))

# Dates in final_holiday_df but not in course_data_df
only_in_holiday = holiday_dates - course_dates
print("\nDates in final_holiday_df but NOT in course_data_df:")
print(sorted(only_in_holiday))

print(f"Number of missing datapoints = {len(only_in_holiday)}")


0   2018-11-27
Name: Date, dtype: datetime64[ns]
2244   2025-04-28
Name: Date, dtype: datetime64[ns]
course_data_df.shape=(2245, 23)
First date in final_holiday_df = 0   2018-11-01
Name: Date, dtype: datetime64[ns]
Last date in final_holiday_df  = 2370   2025-04-28
Name: Date, dtype: datetime64[ns]
final_holiday_df.shape=(2371, 11)
<DatetimeArray>
['2018-11-01 00:00:00', '2018-11-02 00:00:00', '2018-11-03 00:00:00',
 '2018-11-04 00:00:00', '2018-11-05 00:00:00', '2018-11-06 00:00:00',
 '2018-11-07 00:00:00', '2018-11-08 00:00:00', '2018-11-09 00:00:00',
 '2018-11-10 00:00:00',
 ...
 '2025-04-19 00:00:00', '2025-04-20 00:00:00', '2025-04-21 00:00:00',
 '2025-04-22 00:00:00', '2025-04-23 00:00:00', '2025-04-24 00:00:00',
 '2025-04-25 00:00:00', '2025-04-26 00:00:00', '2025-04-27 00:00:00',
 '2025-04-28 00:00:00']
Length: 2371, dtype: datetime64[ns]
Dates in course_data_df but NOT in final_holiday_df:
[]

Dates in final_holiday_df but NOT in course_data_df:
[Timestamp('2018-11-01 00:00:00

In [157]:
# put your DataFrames in a list
all_dfs = [
    guest_df,
    calendar_df,
    weather_df,
    school_holidays_bool_df,
    final_holiday_df,
    course_data_df,
]

# 1. Start with the Date index from the first df
common_idx = pd.DatetimeIndex(all_dfs[0]["Date"].unique())

# 2. Iteratively intersect with each subsequent df’s Date index
for df in all_dfs[1:]:
    common_idx = common_idx.intersection(pd.DatetimeIndex(df["Date"].unique()))

# common_idx now holds only the dates present in _every_ DataFrame

# 3. Filter each DataFrame to that common date-set
for i, df in enumerate(all_dfs):
    all_dfs[i] = df[df["Date"].isin(common_idx)].reset_index(drop=True)

# unpack back
guest_df, calendar_df, weather_df, school_holidays_bool_df, final_holiday_df, course_data_df = all_dfs

# 4. Sanity-check: they should all have the same shape and date range
df_names = ["guest", "calendar", "weather", "school_hols", "final_holiday", "course"]
for name, df in zip(df_names, all_dfs):
    print(f"{name:15s} -> {df.shape[0]} rows, {df['Date'].min().date()}–{df['Date'].max().date()}")

guest           -> 2245 rows, 2018-11-27–2025-04-28
calendar        -> 2245 rows, 2018-11-27–2025-04-28
weather         -> 2245 rows, 2018-11-27–2025-04-28
school_hols     -> 2245 rows, 2018-11-27–2025-04-28
final_holiday   -> 2245 rows, 2018-11-27–2025-04-28
course          -> 2245 rows, 2018-11-27–2025-04-28


In [158]:
for df in all_dfs:
    # print(df.columns)
    print(df.head(1))

        Date GUESTS
0 2018-11-27      0
        Date  is_Friday  is_Monday  is_Saturday  is_Sunday  is_Thursday  \
0 2018-11-27          0          0            0          0            0   

   is_Tuesday  is_Wednesday  
0           1             0  
        Date  tempmax  tempmin  temp  feelslikemax  feelslikemin  feelslike  \
0 2018-11-27      4.0     -0.1   1.7           1.3          -4.4       -1.6   

   humidity  precip  precipprob  windgust  windspeed  cloudcover  \
0      85.1     0.0           0      30.9       15.5        58.0   

   solarradiation  uvindex  rain  snow  
0            23.5        1     0     0  
        Date  IsHoliday
0 2018-11-27          0
        Date  Ascension Day  Christmas  Day of German Unity  Easter Monday  \
0 2018-11-27              0          0                    0              0   

   Good Friday  King's Day  May Day  New Year's Day  Second Christmas Day  \
0            0           0        0               0                     0   

   Whit Mon

In [159]:
COVID_WINDOWS = [
    ('2020-03-01', '2020-05-31'),
    ('2020-12-01', '2021-06-30'),
    ('2021-11-01', '2022-01-31'),
]

def in_covid_period(s, windows=COVID_WINDOWS):
    """Boolean mask: True if s (datetime Series) falls in any covid window."""
    mask = pd.Series(False, index=s.index)
    for start, end in windows:
        mask |= s.between(start, end)
    return mask

# filter guest_df 
guest_df['GUESTS'] = pd.to_numeric(guest_df['GUESTS'], errors='coerce')

guest_df = guest_df[
    (guest_df['GUESTS'].between(1, 400))      # keep 1–400
    & (~in_covid_period(guest_df['Date']))     # drop covid dates
].reset_index(drop=True)

valid_dates = set(guest_df['Date'])

# filter the others
other_dfs = [
    ('calendar',        calendar_df),
    ('weather',         weather_df),
    ('school_holidays', school_holidays_bool_df),
    ('final_holiday',   final_holiday_df),
    ('course',          course_data_df),
]

filtered = {}
for name, df in other_dfs:
    df = df[
        df['Date'].isin(valid_dates)          # align to guest_df dates
        & (~in_covid_period(df['Date']))      # remove covid dates
    ].reset_index(drop=True)
    filtered[name] = df                       # save back

# unpack
calendar_df        = filtered['calendar']
weather_df         = filtered['weather']
school_holidays_bool_df = filtered['school_holidays']
final_holiday_df   = filtered['final_holiday']
course_data_df     = filtered['course']

# sanity check 
for nm, df in [('guest', guest_df), *filtered.items()]:
    print(f'{nm:16s} → {df.shape[0]} rows, {df.Date.min().date()} – {df.Date.max().date()}')

guest            → 1777 rows, 2019-01-01 – 2025-04-28
calendar         → 1777 rows, 2019-01-01 – 2025-04-28
weather          → 1777 rows, 2019-01-01 – 2025-04-28
school_holidays  → 1777 rows, 2019-01-01 – 2025-04-28
final_holiday    → 1777 rows, 2019-01-01 – 2025-04-28
course           → 1777 rows, 2019-01-01 – 2025-04-28


In [160]:
import os

# Define where to save
output_dir = "../data/cleaned/"
os.makedirs(output_dir, exist_ok=True)

# Save each DataFrame individually
dfs = {
    'guest':                guest_df,
    'calendar':             calendar_df,
    'weather':              weather_df,
    'school_holidays':      school_holidays_bool_df,
    'public_holidays':      final_holiday_df,
    'course_sales':         course_data_df,
}

for name, df in dfs.items():
    df.to_csv(f"{output_dir}{name}.csv", index=False)

# Merge them all into one
#    Start from guest_df so we keep only dates with actual guests
full_df = guest_df.copy()

for name, df in dfs.items():
    if name == 'guest':
        continue
    full_df = full_df.merge(df, on='Date', how='left')

# Reorder columns so Date comes first
cols = ['Date'] + [c for c in full_df.columns if c != 'Date']
full_df = full_df[cols]

# Save the merged table
full_df.to_csv(f"{output_dir}full_restaurant_data.csv", index=False)


In [161]:
print(full_df.shape)

(1777, 58)
